In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# 1. 파라미터 및 자동 날짜 설정
tickers = ['BTC-USD', '^IXIC'] # 비트코인(USD), 나스닥 지수

# 코드를 돌리는 날 기준으로 어제 날짜 계산
today = datetime.today()
end_date_obj = today - timedelta(days=1)

# 어제 날짜 기준으로 정확히 2달 전 날짜 계산
start_date_obj = end_date_obj - relativedelta(months=2)

# yfinance가 인식할 수 있는 문자열(YYYY-MM-DD) 형태로 변환
end_date = end_date_obj.strftime('%Y-%m-%d')
start_date = start_date_obj.strftime('%Y-%m-%d')

rolling_window = 30 # 30일 이동 상관계수

# 2. 데이터 수집 (야후 파이낸스)
data = yf.download(tickers, start=start_date, end=end_date)['Close']

# 3. 데이터 전처리
# 나스닥 주말/공휴일 휴장으로 인한 결측치(NaN) 처리
data = data.dropna()

# 4. 설정 기간 상관계수 계산
total_corr = data['BTC-USD'].corr(data['^IXIC'])

# 5. 30일 롤링(이동) 상관계수 계산
rolling_corr = data['BTC-USD'].rolling(window=rolling_window).corr(data['^IXIC'])

# 6. 최근 상관계수 확인
latest_corr = rolling_corr.iloc[-1]

export_data = rolling_corr.dropna().reset_index()
export_data.columns = ['Date', 'Rolling_Correlation']
export_data.to_csv('nasdaq_btc_correlation.csv', index=False)